<a href="https://colab.research.google.com/github/robertnathe/AutonomousCodingAgent/blob/main/jepa__cifar10_pipeline_04.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

jepa_cifar10_pipeline_04.ipynb

Download the CIFAR-10 and extract a fresh copy of the dataset.

In [11]:
# Download and extract to ensure files are in the right place
%cd /content/
!wget -q https://www.cs.toronto.edu/~kriz/cifar-10-python.tar.gz -O cifar-10-python.tar.gz
!tar -xzf cifar-10-python.tar.gz
!mv cifar-10-batches-py/* .
!rm -rf cifar-10-batches-py cifar-10-python.tar.gz

print("Contents of /content/ after extraction:")
!ls -F /content/

/content
Contents of /content/ after extraction:
batches.meta  data_batch_2  data_batch_4  readme.html	test_batch
data_batch_1  data_batch_3  data_batch_5  sample_data/


In [13]:
# Verify the path for the first batch file specifically
test_path = '/content/data_batch_1'
if os.path.exists(test_path):
    print(f"SUCCESS: Found {test_path}")
else:
    print(f"FAILURE: {test_path} still not found. Searching for it...")
    !find /content/ -name "data_batch_1"

SUCCESS: Found /content/data_batch_1


JEPA CIFAR-10 Python Program

In [ ]:
import os
import pickle
import random
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.checkpoint import checkpoint
from torch.utils.data import DataLoader

def unpickle(file):
    with open(file, "rb") as fo:
        return pickle.load(fo, encoding="bytes")

TOTAL_EPOCHS = 150
LINEAR_PROBE_EPOCHS = 20
FEATURE_BATCH_SIZE = 128
LINEAR_PROBE_BATCH_SIZE = 256
LINEAR_PROBE_LR = 1e-3
LINEAR_PROBE_WEIGHT_DECAY = 1e-4
SEED = 42
CIFAR_MEAN = (0.4914, 0.4822, 0.4465)
CIFAR_STD = (0.2470, 0.2435, 0.2616)

def get_sinusoidal_pe(num_patches, embed_dim):
    pe = torch.zeros(num_patches, embed_dim)
    position = torch.arange(num_patches).unsqueeze(1).float()
    div_term = torch.exp(
        torch.arange(0, embed_dim, 2).float()
        * (-np.log(10000.0) / embed_dim)
    )
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe

class JEPADataset(torch.utils.data.Dataset):
    def __init__(
        self,
        data,
        patch_size=4,
        block_size=2,
        min_blocks=1,
        max_blocks=4,
    ):
        self.data = data
        self.P = patch_size
        self.grid = 32 // self.P
        self.num_patches = self.grid**2
        self.block_size = block_size
        self.min_blocks = min_blocks
        self.max_blocks = max_blocks
        self.current_blocks = float(min_blocks)
        self.mean = torch.tensor(CIFAR_MEAN).view(3, 1, 1)
        self.std = torch.tensor(CIFAR_STD).view(3, 1, 1)

    def set_epoch(self, epoch, max_epochs):
        if max_epochs <= 0:
            raise ValueError("max_epochs must be positive")
        progress = min(max(epoch / max_epochs, 0.0), 1.0)
        alpha = 0.5 * (1.0 - np.cos(np.pi * progress))
        self.current_blocks = self.min_blocks + alpha * (
            self.max_blocks - self.min_blocks
        )

    def _sample_num_blocks(self):
        floor_val = int(np.floor(self.current_blocks))
        ceil_val = int(np.ceil(self.current_blocks))
        if floor_val == ceil_val:
            return floor_val
        frac = self.current_blocks - floor_val
        return ceil_val if np.random.rand() < frac else floor_val

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        img = (self.data[idx] - self.mean) / self.std
        all_blocks = [
            (r, c)
            for r in range(self.grid - self.block_size + 1)
            for c in range(self.grid - self.block_size + 1)
        ]
        np.random.shuffle(all_blocks)
        occupied = np.zeros(
            (self.grid, self.grid),
            dtype=bool,
        )
        chosen_blocks = []
        num_blocks = self._sample_num_blocks()
        for r, c in all_blocks:
            block_occupied = occupied[
                r:r + self.block_size,
                c:c + self.block_size,
            ]
            if not block_occupied.any():
                occupied[
                    r:r + self.block_size,
                    c:c + self.block_size,
                ] = True
                chosen_blocks.append((r, c))
                if len(chosen_blocks) >= num_blocks:
                    break
        if len(chosen_blocks) == 0:
            chosen_blocks = [(0, 0)]
        target_pos = []
        for r, c in chosen_blocks:
            for dr in range(self.block_size):
                for dc in range(self.block_size):
                    target_pos.append(
                        (r + dr) * self.grid + (c + dc)
                    )
        target_pos = torch.tensor(
            target_pos,
            dtype=torch.long,
        )
        all_pos = torch.arange(self.num_patches)
        mask = torch.ones(
            self.num_patches,
            dtype=torch.bool,
        )
        mask[target_pos] = False
        context_pos = all_pos[mask]
        return img, context_pos, target_pos

def jepa_collate(batch):
    imgs, ctx_pos_list, tgt_pos_list = zip(*batch)
    batch_size = len(batch)
    imgs = torch.stack(imgs, dim=0)
    max_ctx_len = max(
        x.size(0)
        for x in ctx_pos_list
    )
    ctx_pos = torch.zeros(
        batch_size,
        max_ctx_len,
        dtype=torch.long,
    )
    ctx_valid = torch.zeros(
        batch_size,
        max_ctx_len,
        dtype=torch.bool,
    )
    for i, positions in enumerate(ctx_pos_list):
        length = positions.size(0)
        ctx_pos[i, :length] = positions
        ctx_valid[i, :length] = True
    max_tgt_len = max(
        x.size(0)
        for x in tgt_pos_list
    )
    tgt_pos = torch.zeros(
        batch_size,
        max_tgt_len,
        dtype=torch.long,
    )
    tgt_valid = torch.zeros(
        batch_size,
        max_tgt_len,
        dtype=torch.bool,
    )
    for i, positions in enumerate(tgt_pos_list):
        length = positions.size(0)
        tgt_pos[i, :length] = positions
        tgt_valid[i, :length] = True
    return (
        imgs,
        ctx_pos,
        tgt_pos,
        ctx_valid,
        tgt_valid,
    )

class PatchEmbed(nn.Module):
    def __init__(
        self,
        patch_size=4,
        in_chans=3,
        embed_dim=192,
    ):
        super().__init__()
        self.proj = nn.Conv2d(
            in_chans,
            embed_dim,
            kernel_size=patch_size,
            stride=patch_size,
        )

    def forward(self, x):
        return self.proj(x).flatten(2).transpose(1, 2)

class ContextEncoder(nn.Module):
    def __init__(
        self,
        patch_size=4,
        embed_dim=192,
        depth=4,
        num_heads=4,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            patch_size,
            3,
            embed_dim,
        )
        self.register_buffer(
            "pos_embed",
            get_sinusoidal_pe(64, embed_dim),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth,
            norm=nn.LayerNorm(embed_dim),
            enable_nested_tensor=False,
        )

    def forward(
        self,
        images,
        positions,
        valid_mask=None,
    ):
        emb = self.patch_embed(images)
        emb = emb + self.pos_embed.unsqueeze(0)
        ctx_emb = torch.gather(
            emb,
            1,
            positions.unsqueeze(-1).expand(
                -1,
                -1,
                emb.size(-1),
            ),
        )
        padding_mask = (
            ~valid_mask
            if valid_mask is not None
            else None
        )
        if self.training and ctx_emb.requires_grad:
            if padding_mask is None:
                return checkpoint(
                    self.transformer,
                    ctx_emb,
                    use_reentrant=False,
                )
            return checkpoint(
                lambda x: self.transformer(
                    x,
                    src_key_padding_mask=padding_mask,
                ),
                ctx_emb,
                use_reentrant=False,
            )
        if padding_mask is None:
            return self.transformer(ctx_emb)
        return self.transformer(
            ctx_emb,
            src_key_padding_mask=padding_mask,
        )

class TargetEncoder(nn.Module):
    def __init__(
        self,
        patch_size=4,
        embed_dim=192,
        depth=4,
        num_heads=4,
    ):
        super().__init__()
        self.patch_embed = PatchEmbed(
            patch_size,
            3,
            embed_dim,
        )
        self.register_buffer(
            "pos_embed",
            get_sinusoidal_pe(64, embed_dim),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth,
            norm=nn.LayerNorm(embed_dim),
            enable_nested_tensor=False,
        )

    def forward(self, images):
        emb = self.patch_embed(images)
        emb = emb + self.pos_embed.unsqueeze(0)
        return self.transformer(emb)

def get_momentum(
    epoch,
    max_epochs,
    base_momentum=0.996,
    final_momentum=1.0,
):
    return final_momentum - (
        final_momentum - base_momentum
    ) * (
        np.cos(np.pi * epoch / max_epochs) + 1.0
    ) / 2.0

def update_target_encoder(
    ctx_enc,
    tgt_enc,
    momentum,
):
    with torch.no_grad():
        for p_ctx, p_tgt in zip(
            ctx_enc.parameters(),
            tgt_enc.parameters(),
        ):
            p_tgt.data.mul_(momentum).add_(
                p_ctx.data,
                alpha=1.0 - momentum,
            )

class Predictor(nn.Module):
    def __init__(
        self,
        embed_dim=192,
        depth=2,
        num_heads=4,
    ):
        super().__init__()
        self.mask_token = nn.Parameter(
            torch.zeros(1, 1, embed_dim)
        )
        self.register_buffer(
            "pos_embed",
            get_sinusoidal_pe(64, embed_dim),
        )
        encoder_layer = nn.TransformerEncoderLayer(
            d_model=embed_dim,
            nhead=num_heads,
            batch_first=True,
            norm_first=True,
        )
        self.transformer = nn.TransformerEncoder(
            encoder_layer,
            num_layers=depth,
            norm=nn.LayerNorm(embed_dim),
            enable_nested_tensor=False,
        )
        self.mlp = nn.Sequential(
            nn.Linear(
                embed_dim,
                embed_dim * 4,
            ),
            nn.GELU(),
            nn.Linear(
                embed_dim * 4,
                embed_dim,
            ),
        )
        nn.init.xavier_uniform_(
            self.mlp[0].weight
        )
        nn.init.zeros_(
            self.mlp[0].bias
        )
        nn.init.xavier_uniform_(
            self.mlp[2].weight
        )
        nn.init.zeros_(
            self.mlp[2].bias
        )
        self.output_norm = nn.LayerNorm(
            embed_dim
        )

    def forward(
        self,
        context_latent,
        context_positions,
        target_positions,
        context_valid_mask=None,
        target_valid_mask=None,
    ):
        batch_size, num_target_positions = (
            target_positions.shape
        )
        mask_tokens = self.mask_token.expand(
            batch_size,
            num_target_positions,
            -1,
        )
        target_queries = (
            mask_tokens
            + self.pos_embed[target_positions]
        )
        x = torch.cat(
            [
                context_latent,
                target_queries,
            ],
            dim=1,
        )
        if (
            context_valid_mask is None
            or target_valid_mask is None
        ):
            combined_valid_mask = None
        else:
            combined_valid_mask = torch.cat(
                [
                    context_valid_mask,
                    target_valid_mask,
                ],
                dim=1,
            )
        if combined_valid_mask is None:
            if self.training and x.requires_grad:
                x = checkpoint(
                    self.transformer,
                    x,
                    use_reentrant=False,
                )
            else:
                x = self.transformer(x)
        else:
            padding_mask = ~combined_valid_mask
            if self.training and x.requires_grad:
                x = checkpoint(
                    lambda z: self.transformer(
                        z,
                        src_key_padding_mask=padding_mask,
                    ),
                    x,
                    use_reentrant=False,
                )
            else:
                x = self.transformer(
                    x,
                    src_key_padding_mask=padding_mask,
                )
        x = x[:, -num_target_positions:, :]
        residual = x
        x = self.output_norm(x)
        x = residual + self.mlp(x)
        return x

def get_rng_state():
    state = {
        "python": random.getstate(),
        "numpy": np.random.get_state(),
        "torch": torch.get_rng_state(),
    }
    if torch.cuda.is_available():
        state["cuda"] = torch.cuda.get_rng_state_all()
    return state

def set_rng_state(state):
    if state is None:
        return
    if "python" in state:
        random.setstate(state["python"])
    if "numpy" in state:
        np.random.set_state(state["numpy"])
    if "torch" in state:
        torch.set_rng_state(state["torch"])
    if (
        torch.cuda.is_available()
        and "cuda" in state
    ):
        torch.cuda.set_rng_state_all(
            state["cuda"]
        )

def save_checkpoint(
    path,
    epoch,
    ctx_enc,
    tgt_enc,
    pred,
    optimizer,
    scheduler,
    scaler,
    last_loss,
):
    checkpoint_state = {
        "epoch": epoch,
        "context_encoder": ctx_enc.state_dict(),
        "target_encoder": tgt_enc.state_dict(),
        "predictor": pred.state_dict(),
        "optimizer": optimizer.state_dict(),
        "scheduler": scheduler.state_dict(),
        "scaler": scaler.state_dict(),
        "last_loss": last_loss,
        "rng_state": get_rng_state(),
    }
    temp_path = path + ".tmp"
    torch.save(
        checkpoint_state,
        temp_path,
    )
    os.replace(
        temp_path,
        path,
    )

def load_checkpoint(
    path,
    ctx_enc,
    tgt_enc,
    pred,
    optimizer,
    scheduler,
    scaler,
    device,
):
    checkpoint_state = torch.load(
        path,
        map_location=device,
        weights_only=False,
    )
    ctx_enc.load_state_dict(
        checkpoint_state["context_encoder"]
    )
    tgt_enc.load_state_dict(
        checkpoint_state["target_encoder"]
    )
    pred.load_state_dict(
        checkpoint_state["predictor"]
    )
    optimizer.load_state_dict(
        checkpoint_state["optimizer"]
    )
    scheduler.load_state_dict(
        checkpoint_state["scheduler"]
    )
    scaler.load_state_dict(
        checkpoint_state["scaler"]
    )
    set_rng_state(
        checkpoint_state.get("rng_state")
    )
    return (
        int(checkpoint_state["epoch"]),
        checkpoint_state.get("last_loss"),
    )

def set_global_seed(seed):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)

def load_cifar10(data_dir):
    train_data = []
    train_labels = []
    for i in range(1, 6):
        batch = unpickle(
            os.path.join(
                data_dir,
                f"data_batch_{i}",
            )
        )
        train_data.append(
            batch[b"data"]
        )
        train_labels.extend(
            batch[b"labels"]
        )
    train_data = (
        np.concatenate(train_data)
        .reshape(-1, 3, 32, 32)
        .astype(np.float32)
        / 255.0
    )
    test_batch = unpickle(
        os.path.join(
            data_dir,
            "test_batch",
        )
    )
    test_data = (
        test_batch[b"data"]
        .reshape(-1, 3, 32, 32)
        .astype(np.float32)
        / 255.0
    )
    return (
        torch.from_numpy(train_data),
        torch.tensor(
            train_labels,
            dtype=torch.long,
        ),
        torch.from_numpy(test_data),
        torch.tensor(
            test_batch[b"labels"],
            dtype=torch.long,
        ),
    )

def save_final_model(
    final_model_path,
    ctx_enc,
    tgt_enc,
    pred,
):
    torch.save(
        {
            "context_encoder": ctx_enc.state_dict(),
            "target_encoder": tgt_enc.state_dict(),
            "predictor": pred.state_dict(),
        },
        final_model_path,
    )
    print(
        f"[INFO] Final model weights saved to "
        f"'{final_model_path}'."
    )

def build_jepa_components(
    device,
    train_data,
):
    dataset = JEPADataset(
        train_data
    )
    loader = DataLoader(
        dataset,
        batch_size=64,
        shuffle=True,
        collate_fn=jepa_collate,
        num_workers=0,
        pin_memory=torch.cuda.is_available(),
    )
    ctx_enc = ContextEncoder().to(device)
    tgt_enc = TargetEncoder().to(device)
    pred = Predictor().to(device)
    tgt_enc.load_state_dict(
        ctx_enc.state_dict(),
        strict=False,
    )
    for param in tgt_enc.parameters():
        param.requires_grad = False
    optimizer = torch.optim.AdamW(
        list(ctx_enc.parameters())
        + list(pred.parameters()),
        lr=1e-3,
        weight_decay=0.04,
    )
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr=1e-3,
        steps_per_epoch=len(loader),
        epochs=TOTAL_EPOCHS,
    )
    scaler = torch.amp.GradScaler(
        "cuda",
        enabled=device.type == "cuda",
    )
    return (
        dataset,
        loader,
        ctx_enc,
        tgt_enc,
        pred,
        optimizer,
        scheduler,
        scaler,
    )

def train_or_resume_jepa(
    train_data,
    device,
    checkpoint_path,
    final_model_path,
):
    (
        dataset,
        loader,
        ctx_enc,
        tgt_enc,
        pred,
        optimizer,
        scheduler,
        scaler,
    ) = build_jepa_components(
        device,
        train_data,
    )
    start_epoch = 0
    last_loss = None
    if os.path.exists(
        checkpoint_path
    ):
        print(
            f"[INFO] Found checkpoint: "
            f"{checkpoint_path}"
        )
        try:
            (
                completed_epochs,
                last_loss,
            ) = load_checkpoint(
                checkpoint_path,
                ctx_enc,
                tgt_enc,
                pred,
                optimizer,
                scheduler,
                scaler,
                device,
            )
            start_epoch = completed_epochs
            if start_epoch >= TOTAL_EPOCHS:
                print(
                    f"[INFO] Checkpoint already contains "
                    f"all {TOTAL_EPOCHS} completed epochs."
                )
            else:
                print(
                    f"[INFO] Resuming from epoch "
                    f"{start_epoch + 1}."
                )
            if last_loss is not None:
                print(
                    f"[INFO] Previous epoch loss: "
                    f"{last_loss:.6f}"
                )
        except Exception as exc:
            print(
                f"[WARNING] Checkpoint could not be "
                f"loaded: {exc}"
            )
            print(
                "[WARNING] Starting a new training run."
            )
            start_epoch = 0
            last_loss = None
    else:
        print(
            "[INFO] No checkpoint found. "
            "Starting from epoch 1."
        )
    try:
        for epoch in range(
            start_epoch,
            TOTAL_EPOCHS,
        ):
            dataset.set_epoch(
                epoch,
                TOTAL_EPOCHS,
            )
            ctx_enc.train()
            pred.train()
            tgt_enc.eval()
            total_loss = 0.0
            for (
                imgs,
                ctx_pos,
                tgt_pos,
                ctx_valid,
                tgt_valid,
            ) in loader:
                imgs = imgs.to(
                    device,
                    non_blocking=True,
                )
                ctx_pos = ctx_pos.to(
                    device,
                    non_blocking=True,
                )
                tgt_pos = tgt_pos.to(
                    device,
                    non_blocking=True,
                )
                ctx_valid = ctx_valid.to(
                    device,
                    non_blocking=True,
                )
                tgt_valid = tgt_valid.to(
                    device,
                    non_blocking=True,
                )
                optimizer.zero_grad(
                    set_to_none=True
                )
                with torch.amp.autocast(
                    device_type=device.type,
                    enabled=device.type == "cuda",
                ):
                    c_latent = ctx_enc(
                        imgs,
                        ctx_pos,
                        ctx_valid,
                    )
                    with torch.no_grad():
                        full_t_latent = tgt_enc(
                            imgs
                        )
                        t_latent = torch.gather(
                            full_t_latent,
                            1,
                            tgt_pos.unsqueeze(-1).expand(
                                -1,
                                -1,
                                full_t_latent.size(-1),
                            ),
                        )
                    p_latent = pred(
                        c_latent,
                        ctx_pos,
                        tgt_pos,
                        ctx_valid,
                        tgt_valid,
                    )
                    p_latent = F.normalize(
                        p_latent,
                        dim=-1,
                    )
                    t_latent = F.normalize(
                        t_latent,
                        dim=-1,
                    )
                    per_target_loss = (
                        2.0
                        - 2.0
                        * (
                            p_latent * t_latent
                        ).sum(dim=-1)
                    )
                    valid_float = (
                        tgt_valid.float()
                    )
                    loss = (
                        (
                            per_target_loss
                            * valid_float
                        ).sum()
                        / valid_float.sum().clamp_min(
                            1.0
                        )
                    )
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
                scheduler.step()
                momentum = get_momentum(
                    epoch,
                    TOTAL_EPOCHS,
                )
                update_target_encoder(
                    ctx_enc,
                    tgt_enc,
                    momentum,
                )
                total_loss += loss.item()
            last_loss = (
                total_loss
                / len(loader)
            )
            print(
                f"Epoch {epoch + 1:3d} | "
                f"MaskDifficulty "
                f"{dataset.current_blocks:.3f} | "
                f"Loss {last_loss:.6f}"
            )
            completed_epochs = epoch + 1
            save_checkpoint(
                checkpoint_path,
                completed_epochs,
                ctx_enc,
                tgt_enc,
                pred,
                optimizer,
                scheduler,
                scaler,
                last_loss,
            )
            print(
                f"[INFO] Checkpoint saved after "
                f"epoch {completed_epochs}."
            )
    except KeyboardInterrupt:
        print(
            "\n[INFO] Training interrupted by user."
        )
        print(
            "[INFO] The most recent completed epoch "
            f"is stored in: {checkpoint_path}"
        )
    completed_epochs = start_epoch
    if start_epoch < TOTAL_EPOCHS:
        if os.path.exists(
            checkpoint_path
        ):
            try:
                final_state = torch.load(
                    checkpoint_path,
                    map_location="cpu",
                    weights_only=False,
                )
                completed_epochs = int(
                    final_state["epoch"]
                )
            except Exception:
                completed_epochs = 0
    else:
        completed_epochs = start_epoch
    if completed_epochs >= TOTAL_EPOCHS:
        save_final_model(
            final_model_path,
            ctx_enc,
            tgt_enc,
            pred,
        )
        print(
            "[INFO] Resumable checkpoint remains at "
            f"'{checkpoint_path}'."
        )
    else:
        print(
            "[INFO] Pretraining currently has "
            f"{completed_epochs}/{TOTAL_EPOCHS} "
            "completed epochs."
        )
    return (
        ctx_enc,
        completed_epochs,
    )

@torch.no_grad()
def extract_image_features(
    encoder,
    images,
    device,
    batch_size=FEATURE_BATCH_SIZE,
):
    encoder.eval()
    mean = torch.tensor(
        CIFAR_MEAN,
        dtype=torch.float32,
        device=device,
    ).view(
        1,
        3,
        1,
        1,
    )
    std = torch.tensor(
        CIFAR_STD,
        dtype=torch.float32,
        device=device,
    ).view(
        1,
        3,
        1,
        1,
    )
    feature_batches = []
    for start in range(
        0,
        len(images),
        batch_size,
    ):
        batch = (
            images[
                start:start + batch_size
            ].to(device)
            - mean
        ) / std
        emb = (
            encoder.patch_embed(batch)
            + encoder.pos_embed.unsqueeze(0)
        )
        patch_latent = encoder.transformer(
            emb
        )
        actual_batch_size = (
            patch_latent.size(0)
        )
        grid = patch_latent.view(
            actual_batch_size,
            8,
            8,
            -1,
        )
        global_mean = grid.mean(
            dim=(1, 2)
        )
        top_left = grid[
            :,
            :4,
            :4,
            :,
        ].mean(
            dim=(1, 2)
        )
        top_right = grid[
            :,
            :4,
            4:,
            :,
        ].mean(
            dim=(1, 2)
        )
        bottom_left = grid[
            :,
            4:,
            :4,
            :,
        ].mean(
            dim=(1, 2)
        )
        bottom_right = grid[
            :,
            4:,
            4:,
            :,
        ].mean(
            dim=(1, 2)
        )
        image_features = torch.cat(
            [
                global_mean,
                top_left,
                top_right,
                bottom_left,
                bottom_right,
            ],
            dim=1,
        )
        feature_batches.append(
            image_features.cpu()
        )
    return torch.cat(
        feature_batches,
        dim=0,
    )

def run_linear_probe(
    encoder,
    train_images,
    train_labels,
    test_images,
    test_labels,
    device,
):
    print(
        "\n" + "=" * 60
    )
    print(
        "Frozen CIFAR-10 Linear Probe"
    )
    print(
        "=" * 60
    )
    for param in encoder.parameters():
        param.requires_grad = False
    encoder.eval()
    print(
        "[INFO] Extracting frozen "
        "ContextEncoder representations..."
    )
    train_features = extract_image_features(
        encoder,
        train_images,
        device,
    )
    test_features = extract_image_features(
        encoder,
        test_images,
        device,
    )
    print(
        f"[INFO] Train feature shape: "
        f"{tuple(train_features.shape)}"
    )
    print(
        f"[INFO] Test feature shape:  "
        f"{tuple(test_features.shape)}"
    )
    feat_mean = train_features.mean(
        dim=0,
        keepdim=True,
    )
    feat_std = (
        train_features.std(
            dim=0,
            keepdim=True,
        )
        + 1e-6
    )
    train_features = (
        train_features - feat_mean
    ) / feat_std
    test_features = (
        test_features - feat_mean
    ) / feat_std
    print(
        "[INFO] Training linear classifier..."
    )
    head = nn.Linear(
        train_features.size(1),
        10,
    ).to(device)
    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(
        head.parameters(),
        lr=LINEAR_PROBE_LR,
        weight_decay=LINEAR_PROBE_WEIGHT_DECAY,
    )
    head_dataset = torch.utils.data.TensorDataset(
        train_features,
        train_labels,
    )
    head_loader = DataLoader(
        head_dataset,
        batch_size=LINEAR_PROBE_BATCH_SIZE,
        shuffle=True,
        num_workers=0,
    )
    for epoch in range(
        LINEAR_PROBE_EPOCHS
    ):
        head.train()
        total_loss = 0.0
        for xb, yb in head_loader:
            xb = xb.to(device)
            yb = yb.to(device)
            optimizer.zero_grad(
                set_to_none=True
            )
            logits = head(xb)
            loss = criterion(
                logits,
                yb,
            )
            loss.backward()
            optimizer.step()
            total_loss += (
                loss.item()
                * xb.size(0)
            )
        avg_loss = (
            total_loss
            / len(train_labels)
        )
        print(
            f"Probe Epoch {epoch + 1:2d} | "
            f"Loss {avg_loss:.6f}"
        )
    head.eval()
    with torch.no_grad():
        train_logits = head(
            train_features.to(device)
        )
        train_pred = (
            train_logits.argmax(
                dim=1
            ).cpu()
        )
        train_acc = (
            train_pred == train_labels
        ).float().mean().item()
        test_logits = head(
            test_features.to(device)
        )
        test_pred = (
            test_logits.argmax(
                dim=1
            ).cpu()
        )
        test_acc = (
            test_pred == test_labels
        ).float().mean().item()
    print(
        "\n=== Linear Probe Results ==="
    )
    print(
        f"Train accuracy: "
        f"{train_acc * 100:.2f}%"
    )
    print(
        f"Test accuracy:  "
        f"{test_acc * 100:.2f}%"
    )
    return (
        head,
        train_acc,
        test_acc,
    )

def main():
    set_global_seed(SEED)
    # script_dir = os.path.dirname(
    #     os.path.abspath(__file__)
    # )
    script_dir = "/content/" # Hardcode script_dir for Colab environment
    data_dir = "/content/" # Updated path to look for batch files in /content/
    checkpoint_path = os.path.join(
        script_dir,
        "jepa_checkpoint_04.pth",
    )
    final_model_path = os.path.join(
        script_dir,
        "jepa_pretrained_04.pth",
    )
    device = torch.device(
        "cuda"
        if torch.cuda.is_available()
        else "cpu"
    )
    print(
        f"[INFO] Using device: {device}"
    )
    print(
        "[INFO] Loading CIFAR-10 data..."
    )
    train_images, train_labels, test_images, test_labels = load_cifar10(
        data_dir
    )
    ctx_enc, completed_epochs = train_or_resume_jepa(
        train_images,
        device,
        checkpoint_path,
        final_model_path,
    )
    if completed_epochs < TOTAL_EPOCHS:
        print(
            "[INFO] Linear probe skipped because "
            "pretraining did not complete."
        )
        return
    run_linear_probe(
        ctx_enc,
        train_images,
        train_labels,
        test_images,
        test_labels,
        device,
    )

if __name__ == "__main__":
    main()

[INFO] Using device: cuda
[INFO] Loading CIFAR-10 data...
[INFO] No checkpoint found. Starting from epoch 1.
Epoch   1 | MaskDifficulty 1.000 | Loss 0.227196
[INFO] Checkpoint saved after epoch 1.
Epoch   2 | MaskDifficulty 1.000 | Loss 0.174889
[INFO] Checkpoint saved after epoch 2.


Interpretation of Output